In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import loadmat

In [2]:
CWRU_PATH = Path("../data/raw/CWRU")
files = sorted(CWRU_PATH.glob("*.mat"))

print("Number of files:", len(files))

Number of files: 16


In [3]:
file = files[0]

data = loadmat(file)

print(file.name)
print(data.keys())

B007_0.mat
dict_keys(['__header__', '__version__', '__globals__', 'X118_DE_time', 'X118_FE_time', 'X118_BA_time', 'X118RPM'])


In [4]:
de_keys = [
    key for key in data.keys()
    if "DE_time" in key
]

print(de_keys)

['X118_DE_time']


In [5]:
de_key = de_keys[0]

vibration = data[de_key].squeeze()

print("DE variable:", de_key)
print("Number of samples:", len(vibration))

DE variable: X118_DE_time
Number of samples: 122571


In [6]:
print("Minimum:", np.min(vibration))
print("Maximum:", np.max(vibration))
print("Mean:", np.mean(vibration))
print("Standard deviation:", np.std(vibration))

Minimum: -0.6070200798403194
Maximum: 0.6039338123752496
Mean: 0.012607048538945315
Standard deviation: 0.13866162366530224


In [7]:
print("NaN values:", np.isnan(vibration).sum())

NaN values: 0


In [8]:
print("Infinite values:", np.isinf(vibration).sum())

Infinite values: 0


In [9]:
rpm_keys = [
    key for key in data.keys()
    if "RPM" in key
]

print(rpm_keys)
if rpm_keys:
    rpm = data[rpm_keys[0]].squeeze()
    print("RPM:", rpm)

['X118RPM']
RPM: 1796


In [10]:
results = []

for file in files:

    data = loadmat(file)

    de_keys = [
        key for key in data.keys()
        if "DE_time" in key
    ]

    rpm_keys = [
        key for key in data.keys()
        if "RPM" in key
    ]

    if len(de_keys) == 0:
        print("WARNING: No DE signal:", file.name)
        continue

    vibration = data[de_keys[0]].squeeze()

    if len(rpm_keys) > 0:
        rpm = float(np.asarray(
            data[rpm_keys[0]]
        ).squeeze())
    else:
        rpm = np.nan

    results.append({
        "file": file.name,
        "de_variable": de_keys[0],
        "samples": len(vibration),
        "rpm": rpm,
        "mean": np.mean(vibration),
        "std": np.std(vibration),
        "min": np.min(vibration),
        "max": np.max(vibration),
        "nan_count": np.isnan(vibration).sum(),
        "inf_count": np.isinf(vibration).sum()
    })

In [11]:
verification = pd.DataFrame(results)

verification

,file,de_variable,samples,rpm,mean,std,min,max,nan_count,inf_count
0,B007_0.mat,X118_DE_time,122571,1796.0,0.012607,0.138662,-0.607020,0.603934,0,0
1,B007_1.mat,X119_DE_time,121410,1772.0,0.003892,0.139014,-0.659649,0.639670,0,0
2,B007_2.mat,X120_DE_time,121556,1748.0,0.004564,0.147181,-0.566736,0.604584,0,0
3,B007_3.mat,X121_DE_time,121556,1722.0,0.004200,0.153577,-0.720562,0.671507,0,0
4,IR007_0.mat,X105_DE_time,121265,1797.0,0.013444,0.291216,-1.379886,1.739030,0,0
5,IR007_1.mat,X106_DE_time,121991,1772.0,0.005801,0.292835,-1.402952,1.580819,0,0
6,IR007_2.mat,X107_DE_time,122136,1748.0,0.004552,0.299476,-1.425531,1.639620,0,0
7,IR007_3.mat,X108_DE_time,122917,1721.0,0.004718,0.313571,-1.535499,1.671457,0,0
8,NORMAL_0.mat,X097_DE_time,243938,1796.0,0.012558,0.072687,-0.286638,0.311254,0,0
9,NORMAL_1.mat,X098_DE_time,483903,NaN,0.012564,0.065152,-0.345884,0.317513,0,0


In [12]:
RPM_MAP = {
    0: 1797,
    1: 1772,
    2: 1750,
    3: 1730
}

In [13]:
def get_load(filename):

    suffix = filename.split("_")[-1]

    return int(
        suffix.replace(".mat", "")
    )

In [14]:
verification["load_hp"] = verification["file"].apply(
    get_load
)

verification["expected_rpm"] = verification["load_hp"].map(
    RPM_MAP
)

In [15]:
verification["rpm_difference"] = (
    verification["rpm"]
    - verification["expected_rpm"]
)

In [16]:
verification[
    [
        "file",
        "load_hp",
        "rpm",
        "expected_rpm",
        "rpm_difference"
    ]
]

,file,load_hp,rpm,expected_rpm,rpm_difference
0,B007_0.mat,0,1796.0,1797,-1.0
1,B007_1.mat,1,1772.0,1772,0.0
2,B007_2.mat,2,1748.0,1750,-2.0
3,B007_3.mat,3,1722.0,1730,-8.0
4,IR007_0.mat,0,1797.0,1797,0.0
5,IR007_1.mat,1,1772.0,1772,0.0
6,IR007_2.mat,2,1748.0,1750,-2.0
7,IR007_3.mat,3,1721.0,1730,-9.0
8,NORMAL_0.mat,0,1796.0,1797,-1.0
9,NORMAL_1.mat,1,NaN,1772,NaN


In [17]:
verification.to_csv(
    "../data/processed/CWRU/cwru_signal_verification.csv",
    index=False
)